In [2]:
# Test data repair scripts 

import sys
import os
from dotenv import load_dotenv, find_dotenv
import matplotlib.pyplot as plt
import time
import pandas as pd
import numpy as np
import json

load_dotenv(find_dotenv())

ROOT_PATH = os.getenv("ROOT_PATH")
MY_DATA_PATH = os.getenv("MY_DATA_PATH")
RAW_DATA_PATH = os.getenv("RAW_DATA_PATH")
DEWEY_PATH = os.path.join(RAW_DATA_PATH, "dewey-downloads", "building-permits-united-states")

sys.path.append(os.path.join(ROOT_PATH, "scripts"))
import data_utils as du

sys.path.append(os.path.join(ROOT_PATH, "agent/scripts/ca"))

from data_repair_ca_atascadero import data_repair
MY_JURISDICTION = "Atascadero"

INPUT_FILEPATH = os.path.join(MY_DATA_PATH, "processed_data", "permits_ca_sample.parquet")


In [3]:
df = pd.read_parquet(INPUT_FILEPATH)
sub_df = df[df["JURISDICTION"] == MY_JURISDICTION]

for col in ['FILE_DATE', 'PERMIT_DATE', 'FINAL_DATE']:
    sub_df[f'{col}_FLAG'] = ""

#sub_df_filled = sub_df.copy()
sub_df_filled = data_repair(sub_df)

assert(len(sub_df) == len(sub_df_filled))


In [4]:
print(f"FILE_DATE available (all): {sub_df['FILE_DATE'].notna().mean():.1%} -> {sub_df_filled['FILE_DATE'].notna().mean():.1%}")

print(f"PERMIT_DATE available (all): {sub_df['PERMIT_DATE'].notna().mean():.1%} -> {sub_df_filled['PERMIT_DATE'].notna().mean():.1%}")

print(f"FINAL_DATE available (all): {sub_df['FINAL_DATE'].notna().mean():.1%} -> {sub_df_filled['FINAL_DATE'].notna().mean():.1%}")

mask1 = sub_df['STATUS_NORMALIZED'].isin(['Active', 'Final'])
mask2 = sub_df_filled['STATUS_NORMALIZED'].isin(['Active', 'Final'])
print(f"PERMIT_DATE available (active/final): {sub_df.loc[mask1]['PERMIT_DATE'].notna().mean():.1%} -> {sub_df_filled.loc[mask2]['PERMIT_DATE'].notna().mean():.1%}")

mask1 = sub_df['STATUS_NORMALIZED'].isin(['Final'])
mask2 = sub_df_filled['STATUS_NORMALIZED'].isin(['Final'])
print(f"FINAL_DATE available (final): {sub_df.loc[mask1]['FINAL_DATE'].notna().mean():.1%} -> {sub_df_filled.loc[mask2]['FINAL_DATE'].notna().mean():.1%}")


FILE_DATE available (all): 100.0% -> 100.0%
PERMIT_DATE available (all): 85.5% -> 87.6%
FINAL_DATE available (all): 69.3% -> 69.8%
PERMIT_DATE available (active/final): 94.8% -> 96.9%
FINAL_DATE available (final): 98.6% -> 98.6%


In [5]:
for col in ['STATUS_NORMALIZED', 'FILE_DATE', 'PERMIT_DATE', 'FINAL_DATE']:
    print(sub_df_filled[f'{col}_FLAG'].value_counts())

STATUS_NORMALIZED_FLAG
FIXED    19
Name: count, dtype: int64
Series([], Name: count, dtype: int64)
PERMIT_DATE_FLAG
FILLED    43
Name: count, dtype: int64
FINAL_DATE_FLAG
FILLED    8
Name: count, dtype: int64


In [6]:
print(sub_df['STATUS_NORMALIZED'].value_counts())
print(sub_df_filled['STATUS_NORMALIZED'].value_counts())

STATUS_NORMALIZED
Final        1402
Active        292
Inactive      228
In Review      78
Name: count, dtype: int64
STATUS_NORMALIZED
Final        1415
Active        287
Inactive      228
In Review      70
Name: count, dtype: int64


In [7]:
mask = sub_df_filled["FINAL_DATE"].isna()
#mask = sub_df_filled["JURISDICTION"].notna()
sample = sub_df_filled.loc[mask].sample(1).iloc[0]
DATA = sample["DATA"]
DATES_DATA = du.extract_date_fields(DATA) 

print(f"STATUS_NORMALIZED: {sample['STATUS_NORMALIZED']}    *Filled: {sample['STATUS_NORMALIZED_FLAG']}*")
print(f"RECORD_TYPE_ORIGINAL: {sample['RECORD_TYPE_ORIGINAL']}")
print(f"FILE_DATE: {sample['FILE_DATE']}       *Filled: {sample['FILE_DATE_FLAG']}*")
print(f"PERMIT_DATE: {sample['PERMIT_DATE']}   *Filled: {sample['PERMIT_DATE_FLAG']}*")
print(f"FINAL_DATE: {sample['FINAL_DATE']}     *Filled: {sample['FINAL_DATE_FLAG']}*")

print("DATES_DATA: ")
print(json.dumps(DATES_DATA, indent=2))



STATUS_NORMALIZED: Inactive    *Filled: nan*
RECORD_TYPE_ORIGINAL: BUILDING RESIDENTIAL
FILE_DATE: 2019-06-25       *Filled: nan*
PERMIT_DATE: 2019-07-23   *Filled: nan*
FINAL_DATE: None     *Filled: nan*
DATES_DATA: 
{
  "permit_info": {
    "PermitStatus": "EXPIRED",
    "PermitIssuedDate": "7/23/2019",
    "PermitAppliedDate": "6/25/2019",
    "PermitFinaledDate": "",
    "PermitApprovedDate": "7/23/2019",
    "PermitExpirationDate": "8/29/2020"
  }
}


In [8]:
print("DATA:")
print(json.dumps(json.loads(DATA), indent=2))



DATA:
{
  "fees": {},
  "contacts": [],
  "site_info": {
    "SiteAddr": "11345 ATASCADERO AVE",
    "SiteBlock": "",
    "SiteLotNo": "",
    "SiteTract": "",
    "SiteLotSqFt": "0",
    "ParcelNumber": "045-421-010",
    "PropertyType": "ADDRESS",
    "SectionTwpRng": "",
    "SiteSubdivision": "",
    "SiteCityStateZip": "ATASCADERO, CA, 93422"
  },
  "inspections": [],
  "permit_info": {
    "PermitDesc": "15.0 KW GROUND MOUNT SOLAR W SERVICE PANEL UPGRADE TO 200 A",
    "PermitType": "BUILDING RESIDENTIAL",
    "PermitStatus": "EXPIRED",
    "PermitSubtype": "PHOTOVOLTAIC RESIDENTIAL",
    "PermitIssuedDate": "7/23/2019",
    "PermitAppliedDate": "6/25/2019",
    "PermitFinaledDate": "",
    "PermitApprovedDate": "7/23/2019",
    "PermitExpirationDate": "8/29/2020"
  },
  "search_data": {
    "RECORDID": "AMAN:190625122904262",
    "Site Address": "11345 ATASCADERO AVE",
    "Permit Number": "BRES19-0686"
  }
}
